## 02_Data_Preprocessing_and_Feature_Engineering

The purpose of this notebook is to prepare the Avazu CTR dataset for model building.
The dataset contains no standard missing values and no duplicate impression IDs. Therefore, the preprocessing work focuses on validating the dataset, removing non-predictive identifiers, creating useful time-based features, handling high-cardinality categorical variables, and creating train, validation, and test datasets.

1. Load the raw Avazu dataset.
2. Validate missing values, duplicates, and target values.
3. Remove the impression identifier.
4. Create time-based features from the `hour` column.
5. Identify categorical feature.
6. Create frequency-encoded features for high-cardinality columns.

In [1]:
import os
import sys
import platform
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
train_path = Path("../data/raw/train.gz")
df = pd.read_csv(train_path)
df.head()

,id,click,hour,C1,banner_pos,site_id,site_domain,site_category,app_id,app_domain,...,device_type,device_conn_type,C14,C15,C16,C17,C18,C19,C20,C21
0,1.000009e+18,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,...,1,2,15706,320,50,1722,0,35,-1,79
1,1.000017e+19,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,...,1,0,15704,320,50,1722,0,35,100084,79
2,1.000037e+19,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,...,1,0,15704,320,50,1722,0,35,100084,79
3,1.000064e+19,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,...,1,0,15706,320,50,1722,0,35,100084,79
4,1.000068e+19,0,14102100,1005,1,fe8cc448,9166c161,0569f928,ecad2386,7801e8d9,...,1,0,18993,320,50,2161,0,35,-1,157


In [4]:
# df remains the raw loaded sample.
data = df.copy()
data.head()

,id,click,hour,C1,banner_pos,site_id,site_domain,site_category,app_id,app_domain,...,device_type,device_conn_type,C14,C15,C16,C17,C18,C19,C20,C21
0,1.000009e+18,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,...,1,2,15706,320,50,1722,0,35,-1,79
1,1.000017e+19,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,...,1,0,15704,320,50,1722,0,35,100084,79
2,1.000037e+19,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,...,1,0,15704,320,50,1722,0,35,100084,79
3,1.000064e+19,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,...,1,0,15706,320,50,1722,0,35,100084,79
4,1.000068e+19,0,14102100,1005,1,fe8cc448,9166c161,0569f928,ecad2386,7801e8d9,...,1,0,18993,320,50,2161,0,35,-1,157


## Validate the dataset


In [6]:
# Check missing values
total_missing = int(data.isna().sum().sum())
print("Total missing values:", total_missing)

Total missing values: 0


In [7]:
# Check complete duplicate rows
duplicate_rows = int(data.duplicated().sum())
print("Complete duplicate rows:", duplicate_rows)

Complete duplicate rows: 0


In [9]:
# Check duplicate impression IDs
duplicate_ids = int(data["id"].duplicated().sum())
print("Duplicate impression IDs:", duplicate_ids)

Duplicate impression IDs: 0


In [10]:
# Validate the target
df["click"].unique()

array([0, 1])

## Remove the impression identifier
id uniquely identifies an ad impression. It does not describe the user, app, site, device, or ad context. It has almost one unique value per row, so it is not useful as a predictive feature.

In [11]:
print("Unique IDs:", data["id"].nunique())
print("Rows:", len(data))

Unique IDs: 40428967
Rows: 40428967


In [12]:
data = data.drop(columns=["id"])
print("The id column was removed.")

The id column was removed.


## Parse the raw time column

In [17]:
data["timestamp"] = pd.to_datetime(
    data["hour"].astype(str),
    format="%y%m%d%H",
    errors="coerce"
)

In [15]:
# Validate:
invalid_timestamps = int(data["timestamp"].isna().sum())
print("Invalid timestamps:", invalid_timestamps)
print("Start time:", data["timestamp"].min())
print("End time:", data["timestamp"].max())

Invalid timestamps: 0
Start time: 2014-10-21 00:00:00
End time: 2014-10-30 23:00:00


# Create time-based features

In [18]:
data["day"] = data["timestamp"].dt.day.astype("int8")
data["hour_of_day"] = data["timestamp"].dt.hour.astype("int8")
data["day_of_week"] = data["timestamp"].dt.dayofweek.astype("int8")
data["is_weekend"] = (
data["day_of_week"].isin([5, 6])
).astype("int8")
# pandas considered 5, 6 as saturday, sunday

In [20]:
# Create a time-of-day feature:
def create_time_period(hour):
    if 0 <= hour < 6:
        return "night" 
    elif 6 <= hour < 12:
        return "morning"
    elif 12 <= hour < 18:
        return "afternoon"
    else:
        return "evening"
data["time_period"] = (
    data["hour_of_day"].apply(create_time_period).astype("category")
)

In [21]:
# Inspect:
data[
    ["hour", "timestamp", "day", "hour_of_day", "day_of_week", "is_weekend", "time_period"]
].head()

,hour,timestamp,day,hour_of_day,day_of_week,is_weekend,time_period
0,14102100,2014-10-21,21,0,1,0,night
1,14102100,2014-10-21,21,0,1,0,night
2,14102100,2014-10-21,21,0,1,0,night
3,14102100,2014-10-21,21,0,1,0,night
4,14102100,2014-10-21,21,0,1,0,night


In [22]:
# Remove the raw hour column
# The useful information from hour is now represented by clearer time features.
data = data.drop(columns=["hour"])
print("The raw hour column was removed.")

The raw hour column was removed.


In [24]:
data.shape

(40428967, 28)

## Define categorical feature groups 

In [23]:
string_categorical_columns = ["site_id", "site_domain", "site_category", 
                              "app_id", "app_domain", "app_category",
                              "device_id","device_ip","device_model"
]

integer_categorical_columns = ["C1","banner_pos","device_type","device_conn_type","C14", "C15","C16","C17","C18","C19","C20","C21"
]

## Analyze cardinality

In [26]:
unique_counts = data.nunique(dropna=False).sort_values(ascending=False)
unique_counts

device_ip           6729486
device_id           2686408
app_id                 8552
device_model           8251
site_domain            7745
site_id                4737
C14                    2626
app_domain              559
C17                     435
timestamp               240
C20                     172
C19                      68
C21                      60
app_category             36
site_category            26
hour_of_day              24
day                      10
C16                       9
C15                       8
banner_pos                7
day_of_week               7
C1                        7
device_type               5
time_period               4
C18                       4
device_conn_type          4
is_weekend                2
click                     2
dtype: int64

In [31]:
def cardinality_type(unique_counts):
    if unique_counts <= 20:
        return "Low"
    elif unique_counts <= 100:
        return "Medium"
    else:
        return "High"
      

In [32]:
cardinality = unique_counts.apply(cardinality_type)
cardinality

device_ip             High
device_id             High
app_id                High
device_model          High
site_domain           High
site_id               High
C14                   High
app_domain            High
C17                   High
timestamp             High
C20                   High
C19                 Medium
C21                 Medium
app_category        Medium
site_category       Medium
hour_of_day         Medium
day                    Low
C16                    Low
C15                    Low
banner_pos             Low
day_of_week            Low
C1                     Low
device_type            Low
time_period            Low
C18                    Low
device_conn_type       Low
is_weekend             Low
click                  Low
dtype: str